In [1]:
import pandas as pd
import warnings
import plotly.express as px
import numpy as np

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
import plotly.io as pio

pio.renderers.default = "svg"

In [3]:
df = pd.read_parquet(r"https://github.com/monatagelsir7/enivornmental_impact_of_aviation/raw/refs/heads/main/compressed_aviation_traffic_data.parquet")
df

,airline_iata,acft_icao,acft_class,seymour_proxy,source,seats,n_flights,iata_departure,iata_arrival,departure_lon,...,arrival_country,arrival_continent,seats_no_est_scaling,distance_km,ask,rpk,fuel_burn_seymour,fuel_burn,co2,domestic
0,1SQ,PA32,PP,P28B,BTS,62112.0,11466.5,SPN,TIQ,145.729004,...,MP,OC,62112.0,17.765225,1.103434e+06,9.092294e+05,20.486379,2.349071e+05,7.423063e+05,1
1,1SQ,PA32,PP,P28B,BTS,62112.0,11466.5,TIQ,SPN,145.619003,...,MP,OC,62112.0,17.765225,1.103434e+06,9.092294e+05,20.486379,2.349071e+05,7.423063e+05,1
2,HA,B712,NB,B712,BTS,1246720.0,9740.0,HNL,OGG,-157.924228,...,US,OC,1246720.0,162.001417,2.019704e+08,1.664236e+08,1305.895783,1.271942e+07,4.019338e+07,1
3,HA,B712,NB,B712,BTS,1246720.0,9740.0,OGG,HNL,-156.431212,...,US,OC,1246720.0,162.001417,2.019704e+08,1.664236e+08,1305.895783,1.271942e+07,4.019338e+07,1
4,HA,B712,NB,B712,BTS,879296.0,6869.5,HNL,KOA,-157.924228,...,US,OC,879296.0,262.778946,2.310605e+08,1.903938e+08,1567.819046,1.077013e+07,3.403362e+07,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
293833,TTL,None,None,None,ANAC,0.0,NaN,VIX,VIX,-40.285000,...,BR,SA,0.0,0.000000,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,1
293834,None,None,None,None,AUS Stats,0.0,NaN,MEL,ASP,144.843002,...,AU,OC,0.0,1857.153168,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,1
293835,None,None,None,None,AUS Stats,0.0,NaN,SYD,ASP,151.177002,...,AU,OC,0.0,2020.723001,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,1
293836,None,None,None,None,AUS Stats,0.0,NaN,MOV,BNE,148.076996,...,AU,OC,0.0,779.613350,0.000000e+00,0.000000e+00,NaN,0.000000e+00,0.000000e+00,1


In [ ]:
df["log_co2"] = np.log1p(df["co2"])
df["log_distance"] = np.log1p(df["distance_km"])

In [ ]:
df.departure_continent.value_counts()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.columns = df.columns.str.lower().str.replace(" ", "_")

Missing values

In [ ]:
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_percentage = (missing_summary / len(df) * 100).round(2)
missing_df = pd.DataFrame(
    {"Missing Values": missing_summary, "Percent Missing": missing_percentage}
)
missing_df

In [ ]:
missing_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

### Plot distribution of CO2 emissions

In [ ]:
fig = px.histogram(
    df,
    x="co2",
    nbins=100,
    title="Distribution of CO2 emissions(kg)",
    # labels={"log_distance": "Distance (km)"},
    opacity=0.75,
)

fig.update_layout(template="plotly_white", yaxis_title="Frequency", bargap=0.05)

fig.show()

### Let's log transform the CO2 emissions and see the plot again

In [ ]:
fig = px.histogram(
    df,
    x="log_co2",
    nbins=100,
    title="Distribution of log CO2 emissions(kg)",
    # labels={"log_distance": "Distance (km)"},
    opacity=0.75,
)

fig.update_layout(template="plotly_white", yaxis_title="Frequency", bargap=0.05)

fig.show()

### Plot distribution of distance

In [ ]:
fig = px.histogram(
    df,
    x="distance_km",
    nbins=100,
    title="Distribution of Flight Distance",
    # labels={"log_distance": "Distance (km)"},
    opacity=0.75,
)

fig.update_layout(template="plotly_white", yaxis_title="Frequency", bargap=0.05)

fig.show()

Here we can see the distribution of distance is not normal. and concentrated on 0s. Should figure that out. We can try to log transform the distance and see if it helps.

In [ ]:
fig = px.histogram(
    df,
    x="log_distance",
    nbins=100,
    title="Distribution of Log Flight Distance",
    labels={"log_distance": "Log Distance (km)"},
    opacity=0.75,
)

fig.update_layout(template="plotly_white", yaxis_title="Frequency", bargap=0.05)

fig.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

In [ ]:
df

In [ ]:
df.columns

In [ ]:
def distance_cumul_plot_flights(flights_df):
    fig = go.Figure()

    # Define bins for a quick cumulative distribution rendering. 10 km
    bins = list(range(0, int(flights_df["distance_km"].max()) + 10, 10))

    # Cumulative distributions for each metric
    # Seats
    hist_seats, edges_seats = (
        flights_df["seats"].groupby(pd.cut(flights_df["distance_km"], bins)).sum(),
        bins[1:],
    )
    hist_cumul_seats = hist_seats.cumsum() / hist_seats.sum() * 100
    fig.add_trace(
        go.Scatter(
            x=edges_seats,
            y=hist_cumul_seats,
            mode="lines",
            name="Seats",
            line=dict(color="#1f77b4", width=2),
            hovertemplate="%{y:.2f} %",
        )
    )

    # ASK
    hist_ask, edges_ask = (
        flights_df["ask"].groupby(pd.cut(flights_df["distance_km"], bins)).sum(),
        bins[1:],
    )
    hist_cumul_ask = hist_ask.cumsum() / hist_ask.sum() * 100
    fig.add_trace(
        go.Scatter(
            x=edges_ask,
            y=hist_cumul_ask,
            mode="lines",
            name="ASK",
            line=dict(color="#ff7f0e", width=2),
            hovertemplate="%{y:.2f} %",
        )
    )

    #  CO2
    hist_co2, edges_co2 = (
        flights_df["co2"].groupby(pd.cut(flights_df["distance_km"], bins)).sum(),
        bins[1:],
    )
    hist_cumul_co2 = hist_co2.cumsum() / hist_co2.sum() * 100
    fig.add_trace(
        go.Scatter(
            x=edges_co2,
            y=hist_cumul_co2,
            mode="lines",
            name="CO2 (kg)",
            line=dict(color="#2ca02c", width=2),
            hovertemplate="%{y:.2f} %",
        )
    )

    # Formatting
    fig.update_layout(
        title="Metrics cumulative distribution vs flight distance",
        xaxis_title="Distance (km)",
        yaxis_title="Cumulative distribution (%)",
        template="plotly_white",
        hovermode="x",
        margin=dict(l=60, r=60, t=60, b=60),
        legend=dict(
            x=0.82,
            y=0.08,
            bgcolor="rgba(255, 255, 255, 0.5)",
        ),
    )

    return fig

In [ ]:
distance_cumul_plot_flights(df)

1. Seats
Shows that most seating capacity is deployed on flights shorter than 5000 km. (cumulative 90% of all seats). This includes short and medium routes

2. ASK (~65% under 5000 km)
Indicates that ~65% of available seat kilometers (a proxy for supply) also comes from shorter flights.
Suggests that shorter flights dominate volume, though not in direct proportion to seats (because distance matters in ASK).

3. CO2 (~65% under 5000 km)
CO2 emissions track ASK closely — again, showing that shorter flights contribute significantly to total emissions.
However, there's a divergence from seats, meaning short flights emit more per seat compared to long-haul flights.

In [ ]:
def distance_histogram_plot_flights(flights_df, value_watched_flights, strategy="mean"):
    fig = go.Figure()

    # Define bins for the histogram (500 km intervals)
    bin_width = 500
    bins = list(range(0, int(flights_df["distance_km"].max()) + bin_width, bin_width))
    bin_centers = [b + bin_width / 2 for b in bins[:-1]]  # Midpoints of each bin

    bin_ranges = [f"{b - bin_width / 2}-{b + bin_width / 2}" for b in bin_centers]

    # Group by distance bins and calculate the mean or sum of the specified value
    if strategy == "mean":
        grouped = flights_df.groupby(pd.cut(flights_df["distance_km"], bins))[
            value_watched_flights
        ].mean()
    elif strategy == "sum":
        grouped = flights_df.groupby(pd.cut(flights_df["distance_km"], bins))[
            value_watched_flights
        ].sum()

    fig.add_trace(
        go.Bar(
            x=bin_centers,  # Use the center of bins for tick alignment
            y=grouped,
            name=value_watched_flights,
            width=bin_width,
            marker=dict(color="#EE9B00", opacity=0.5),
            hovertemplate=(
                "Distance %{customdata} km:<br>"
                + value_watched_flights
                + " %{y:.2e}<extra></extra>"
            ),
            customdata=bin_ranges,
        )
    )

    # Formatting
    fig.update_layout(
        title=f"Repartition of {strategy.upper()} {value_watched_flights.upper()} by flight distance",
        xaxis_title="Distance (km)",
        yaxis_title=value_watched_flights,
        template="plotly_white",
        hovermode="closest",
        bargap=0.3,
        xaxis=dict(
            tickmode="linear",
            dtick=bin_width,
            range=[0, bins[-1]],
        ),
        margin=dict(l=60, r=60, t=60, b=60),
    )

    return fig

In [ ]:
distance_histogram_plot_flights(df, "co2", "mean")

In [ ]:
distance_histogram_plot_flights(df, "co2", "sum")

Above results show that Shorter flights contribute most to **total** CO2 emissions. 
- The first histogram shows that most of the total CO2 emissions come from flights in the 500–2500 km range, despite being relatively short distances. This happens because short and medium length flights occur far more frequently, leading to cumulative emissions being very high.

Long flights emit more CO2 **per flight**
- The second histogram shows the average CO₂ per flight increases with distance. For flights longer than 10000 km, individual flights emit tens of millions of kg CO2, reflecting the inefficiency of extra long routes.

In [ ]:
distance_histogram_plot_flights(df, "ask")

### Plot distance shared by type of flight (domestic | international)

In [ ]:
def distance_share_dom_int_flights(flights_df, value_watched_flights, strategy="sum"):
    fig = go.Figure()

    bin_width = 500
    bins = list(range(0, int(flights_df["distance_km"].max()) + bin_width, bin_width))
    bin_centers = [b + bin_width / 2 for b in bins[:-1]]
    if strategy == "mean":
        # Group by distance bins and calculate the mean or sum of the specified value
        grouped = (
            flights_df.groupby([pd.cut(flights_df["distance_km"], bins), "domestic"])[
                value_watched_flights
            ]
            .mean()
            .unstack(fill_value=0)
        )
    elif strategy == "sum":
        grouped = (
            flights_df.groupby([pd.cut(flights_df["distance_km"], bins), "domestic"])[
                value_watched_flights
            ]
            .sum()
            .unstack(fill_value=0)
        )

    share_df = grouped.div(grouped.sum(axis=1), axis=0) * 100

    bin_ranges = [f"{b - bin_width / 2}-{b + bin_width / 2}" for b in bin_centers]

    for flight_type in share_df.columns:
        fig.add_trace(
            go.Bar(
                x=bin_centers,
                y=share_df[flight_type],
                name="Domestic" if flight_type == 1 else "International",
                width=bin_width,
                opacity=0.7,
                hovertemplate=(
                    "Distance %{customdata} km:<br>" + "%{y:.2f} %<extra></extra>"
                ),
                customdata=bin_ranges,
                marker=dict(line=dict(width=0)),
            )
        )

    fig.update_layout(
        title=f"Flight type vs flight distance<br>Weighting on: {strategy.upper()} of {value_watched_flights}",
        xaxis_title="Distance (km)",
        yaxis_title="Flight type distribution (%)",
        template="plotly_white",
        hovermode="x",
        barmode="stack",
        yaxis=dict(tickformat=".0f", range=[0, 100]),  # Ensure % scaling
        xaxis=dict(
            tickmode="linear",
            dtick=bin_width,
            range=[0, bins[-1]],
        ),
        legend_title="Flight Type",
        legend=dict(x=0.82, y=0.08, bgcolor="rgba(255, 255, 255, 0.5)"),
        colorway=px.colors.qualitative.T10,
        margin=dict(l=60, r=60, t=60, b=60),
    )

    return fig

#### Looking at the total co2 emissions by distance and type of flight.

In [ ]:
distance_share_dom_int_flights(df, "co2")

Even though long flights are fewer, they are almost exclusively international and they account for the majority of CO2 emissions at longer distances, making them key targets for global climate policy and airline sustainability efforts.

Let's see if the tendency in the previous plots is the same here, differs between average and sum of co2. 

In [ ]:
distance_share_dom_int_flights(df, "co2", "mean")

If we look at the average CO2 emissions, domestic flights contribute significantly to average CO2 emissions at medium distances (2000–8500 km), they *emit more per flight than international flights.*

**This challenges the assumption that domestic flights are always more efficient and suggests they may be less optimized or operate with lower occupancy at these ranges.**

### Plot CO2 vs Seats

In [ ]:
fig = px.scatter(
    df,
    x="seats",
    y="co2",
    opacity=0.3,
    title="CO₂ Emissions vs Number of Seats",
    labels={"seats": "Number of Seats", "co2": "CO₂ Emissions (kg)"},
)

fig.update_traces(marker=dict(color="#EE9B00"))
fig.update_layout(template="plotly_white")
fig.show()

More seats generally mean more emissions but with large variability


### Boxplot: CO2 by Aircraft Class

In [ ]:
fig = px.box(
    df,
    x="acft_class",
    y="co2",
    points="outliers",  # we can use "all", "outliers", or False
    title="CO₂ Emissions by Aircraft Class",
    labels={"co2": "CO₂ Emissions (kg)", "acft_class": "Aircraft Class"},
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Aircraft Class",
    yaxis_title="CO₂ Emissions (kg)",
    margin=dict(l=60, r=60, t=60, b=60),
)

fig.update_xaxes(tickangle=45)

fig.show()

- Wide-body aircraft emit the most on average. Extreme outliers are present in this category.

- Narrow-body  and Regional Jets  follow in the same pattern but with less emissions.

- Private Jets and Piston Propeller aircraft have much lower emissions

### Correlation matrix for CO2 emissions


In [ ]:
numeric_df = df.select_dtypes(include=["float64", "int64"])
correlation_matrix = numeric_df.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix[["co2"]].sort_values(by="co2", ascending=False),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
)
plt.title("Correlation of Numerical Features with CO2 Emissions")
plt.tight_layout()
plt.show()

### Correlation matrix for Log CO2 emissions

### Correlation matrix for Log CO2 emission by other variables

In [ ]:
numeric_df = df.select_dtypes(include=["float64", "int64"])
correlation_matrix = numeric_df.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix[["log_co2"]].sort_values(by="log_co2", ascending=False),
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
)
plt.title("Correlation of Numerical Features with Log CO2 Emissions")
plt.tight_layout()
plt.show()

correlation_matrix_sorted = correlation_matrix.sort_values(
    by="log_co2", ascending=False
)

### Correlation matrix for all variables

In [ ]:
numeric_df = df.select_dtypes(include=["float64", "int64"]).dropna()
correlation_matrix = numeric_df.corr()

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

plt.figure(figsize=(12, 8))
sns.heatmap(
    correlation_matrix,
    mask=mask,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
)
plt.title("Lower Triangle Correlation Matrix of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = [
    "seats",
    "n_flights",
    "seats_no_est_scaling",
    "distance_km",
    "log_distance",
    "ask",
    "rpk",
    "fuel_burn_seymour",
    "fuel_burn",
    "co2",
    "log_co2",
]
numeric_data = df[numeric_cols]

missing_summary = numeric_data.isnull().sum().sort_values(ascending=False)

outlier_info = {}
for col in numeric_cols:
    if col in numeric_data.columns:
        q1 = numeric_data[col].quantile(0.25)
        q3 = numeric_data[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        outliers = (
            (numeric_data[col] < lower_bound) | (numeric_data[col] > upper_bound)
        ).sum()
        pct_outliers = (outliers / len(numeric_data)) * 100
        outlier_info[col] = {
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "n_outliers": outliers,
            "% outliers": round(pct_outliers, 2),
        }

outliers_df = pd.DataFrame(outlier_info).T.sort_values(by="n_outliers", ascending=False)
outliers_df

### Capping them with 99 percentile.


In [ ]:
for col in numeric_cols:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)

Capping values for whole data: 
- **Lower Upper Column**
- 0.0 266362.1484 *seats*
- 0.5 1047.0 *n_flights*
- 0.0 205793.0 *seats_no_est_scaling*
- 0.0 9487.429969576144 *distance_km*
- 0.0 9.157828437408874 *log_distance*
- 0.0 661030796.2811933 *ask*
- 0.0 544689376.1357032 *rpk*
- 0.0 75479.37857206319 *fuel_burn_seymour*
- 0.0 16210973.498874782 *fuel_burn*
- 0.0 51226676.25644431 *co2*
- 0.0 17.75177099448037 *log_co2*

### Fill mising with median

In [ ]:
df.isnull().sum()

In [ ]:
df["co2_per_distance"] = df["co2"] / df["distance_km"]
df

In [ ]:
df.isnull().sum()

In [ ]:
df["co2_per_distance"].fillna(
    0, inplace=True
)  # becuase if co2 = 0, distance = 0, co2_per_distance computed as Nan

In [ ]:
df[numeric_cols] = numeric_data.fillna(numeric_data.median())
df[numeric_cols].isnull().sum()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

### Identify columns where negative or zero values should not logically exist


In [ ]:
df.shape

In [ ]:
positive_only_cols = [
    "distance_km",
    "co2",
    "fuel_burn",
    "ask",
    "rpk",
    "seats",
    "seats_no_est_scaling",
    "co2_per_distance",
]

invalid_counts = {
    col: {"negative_values": (df[col] < 0).sum(), "zero_values": (df[col] == 0).sum()}
    for col in positive_only_cols
}

invalid_df = pd.DataFrame(invalid_counts).T

for col in positive_only_cols:
    df = df[(df[col] > 0)]

# df["log_co2"] = np.log1p(df["co2"])
# df["log_distance"] = np.log1p(df["distance_km"])

df.shape[0]

### Impute categorical variables

We can use countries to fill the continent missing values. 

In [ ]:
continent_missing = df[
    (df["departure_continent"].isna()) & (df["departure_country"].notna())
]["departure_country"].value_counts()

In [ ]:
departure_country_to_continent = (
    df[df["departure_continent"].notna()]
    .groupby("departure_country")["departure_continent"]
    .agg(lambda x: x.value_counts().idxmax())
)
departure_country_to_continent

In [ ]:
arrival_country_to_continent = (
    df[df["arrival_continent"].notna()]
    .groupby("arrival_country")["arrival_continent"]
    .agg(lambda x: x.value_counts().idxmax())
)

In [ ]:
arrival_country_to_continent

In [ ]:
df.isnull().sum()

**There is 125568 missing values in Arrival and Departure Continent**

Impute them with the mapping data we get above

In [ ]:
# # Impute missing values based on country and continent mapping
# df["departure_continent"] = df.apply(
#     lambda row: (
#         departure_country_to_continent[row["departure_country"]]
#         if pd.isna(row["departure_continent"])
#         and row["departure_country"] in departure_country_to_continent
#         else row["departure_continent"]
#     ),
#     axis=1,
# )

# df["arrival_continent"] = df.apply(
#     lambda row: (
#         arrival_country_to_continent[row["arrival_country"]]
#         if pd.isna(row["arrival_continent"])
#         and row["arrival_country"] in arrival_country_to_continent
#         else row["arrival_continent"]
#     ),
#     axis=1,
# )

**After mapping we only have 13825 missing values left.**

In [ ]:
# df.isnull().sum().sort_values(ascending=False)

**Impute any still-missing countries with Unknown**

In [ ]:
df.to_parquet(r"https://github.com/monatagelsir7/enivornmental_impact_of_aviation/raw/refs/heads/main/cleaned_aviation_data_with_outliers_v4.parquet")

In [ ]:
df.departure_continent.value_counts()

In [ ]:
df["departure_country"].fillna("Unknown", inplace=True)
df["arrival_country"].fillna("Unknown", inplace=True)

df["departure_continent"].fillna("Unknown", inplace=True)
df["arrival_continent"].fillna("Unknown", inplace=True)

In [ ]:
df.isnull().sum().sort_values(ascending=False)

Missing all others impute with Unknown, longtitude and lattitude with median

In [ ]:
df["acft_class"].fillna("Unknown", inplace=True)
df["seymour_proxy"].fillna("Unknown", inplace=True)
df["acft_icao"].fillna("Unknown", inplace=True)

df["airline_iata"].fillna("Unknown", inplace=True)

coord_cols = ["departure_lon", "departure_lat", "arrival_lon", "arrival_lat"]
df[coord_cols] = df[coord_cols].fillna(
    df[coord_cols].median()
)  # but not sure if this is correct or not

df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.to_parquet(r"https://github.com/monatagelsir7/enivornmental_impact_of_aviation/raw/refs/heads/main/cleaned_aviation_data_v1.parquet", index=False)

In [ ]:
import pandas as pd
df_v3 = pd.read_parquet("/Users/ilseoplee/enivornmental_impact_of_aviation-2/0.Data_after_cleaning/cleaned_aviation_data_v3.parquet")
df_v3.head(5)